# Моделирование стоимости недвижимости Дубая (ML)

Сравнение Linear / Ridge / RandomForest / LightGBM / MLP на `off_plan.csv`:
- целевая — `price_usd`, обучение на `log1p(price_usd)`;
- временное разделение (train 2021–2023, val 2024, test 2025-01…2026-04);
- эксперимент с шумами (признаки + цены в train);
- сценарный анализ и интерпретация (коэффициенты, permutation importance, worst errors).

**Данные**: `area_prices_monthly.csv`, `metro_stations.csv`, `off_plan.csv` — должны лежать в корневой директории Google Colab (`/content/`).

In [ ]:
# LightGBM устанавливается в текущий runtime Google Colab.
%pip -q install lightgbm

import warnings
warnings.filterwarnings("ignore")

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

from lightgbm import LGBMRegressor, early_stopping

## 1. Конфигурация

In [ ]:
BASE_DIR = Path.cwd()  # В Google Colab это обычно /content
DATA_DIR = BASE_DIR
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

AREA_PRICES_PATH = DATA_DIR / "area_prices_monthly.csv"
METRO_PATH       = DATA_DIR / "metro_stations.csv"
OFFPLAN_PATH     = DATA_DIR / "off_plan.csv"

TARGET_RAW = "price_usd"
TARGET     = "log_price"

TRAIN_END = "2023-12-31"
VAL_END   = "2024-12-31"
TEST_END  = "2026-04-30"

RANDOM_STATE   = 42
NOISE_LEVELS   = (0.05, 0.10, 0.20)
NOISE_REPEATS  = 5

print("Рабочая директория:", BASE_DIR)
print("Папка с результатами:", OUT_DIR)

## 2. Загрузка данных и первичный отчёт

In [ ]:
def load_datasets():
    paths = [AREA_PRICES_PATH, METRO_PATH, OFFPLAN_PATH]
    missing = [path.name for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Не найдены файлы в корневой директории Colab: " + ", ".join(missing)
            + ". Загрузите CSV в /content/ и повторно запустите эту ячейку."
        )

    area  = pd.read_csv(AREA_PRICES_PATH)
    metro = pd.read_csv(METRO_PATH)
    off   = pd.read_csv(OFFPLAN_PATH)
    return area, metro, off


def basic_report(df, name):
    print(f"\n===== {name} =====")
    print(f"shape = {df.shape}")
    print(f"dtypes:\n{df.dtypes.value_counts()}")
    print("\nmissing (top-15):")
    print(df.isna().mean().sort_values(ascending=False).head(15))
    print(f"\nduplicates: {df.duplicated().sum()}")
    num = df.select_dtypes("number")
    if not num.empty:
        print("\nnum describe:")
        print(num.describe().T[["mean", "std", "min", "50%", "max"]])


area, metro, off = load_datasets()
basic_report(area, "area_prices_monthly")
basic_report(metro, "metro_stations")
basic_report(off, "off_plan")

## 3. EDA

In [ ]:
# 1. Распределение цены (до/после лога)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(off["price_usd"], bins=60, ax=ax[0]); ax[0].set_title("price_usd")
sns.histplot(np.log1p(off["price_usd"]), bins=60, ax=ax[1]); ax[1].set_title("log1p(price_usd)")
fig.tight_layout(); fig.savefig(OUT_DIR / "01_price_distribution.png", dpi=120); plt.show()

# 2. Категориальные
for col in ["property_category", "property_type", "developer_tier", "metro_line"]:
    fig, ax = plt.subplots(figsize=(8, 3))
    off[col].value_counts().head(15).plot.bar(ax=ax, rot=45)
    ax.set_title(f"{col} (top-15)"); fig.tight_layout()
    fig.savefig(OUT_DIR / f"02_{col}.png", dpi=120); plt.show()

# 3. Топ сообществ по медианной цене
med = (off.groupby("community")["price_usd"]
          .agg(["median", "count"]).sort_values("median", ascending=False).head(20))
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(med.index[::-1], med["median"].values[::-1])
ax.set_title("Median price by community (top-20)"); fig.tight_layout()
fig.savefig(OUT_DIR / "03_community_median.png", dpi=120); plt.show()

# 4. Динамика
_ = off.copy(); _. _ = None  # noqa
ts = pd.to_datetime(off["date_listed"], errors="coerce")
tmp = off.assign(date_listed=ts).set_index("date_listed").resample("MS")["price_usd"].median()
fig, ax = plt.subplots(figsize=(12, 4))
tmp.plot(ax=ax); ax.set_title("Monthly median price_usd"); fig.tight_layout()
fig.savefig(OUT_DIR / "04_monthly_median.png", dpi=120); plt.show()

# 5. Корреляции
num_cols = ["bedrooms", "area_sqft", "metro_distance_min",
            "to_burj_khalifa_km", "mortgage_rate_at_listing"]
corr = off[num_cols + ["price_usd"]].corr()["price_usd"].sort_values()
print("Корреляция с price_usd:\n", corr)

# 6. Топ-10 станций
print("\nТоп-10 станций по кол-ву объявлений:")
print(off["metro_station"].value_counts().head(10))

# 7. Boxplot цена по категории (в лог-шкале)
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=off, x="property_category", y=np.log1p(off["price_usd"]), ax=ax)
ax.set_title("log1p(price) by property_category"); fig.tight_layout()
fig.savefig(OUT_DIR / "05_price_by_category.png", dpi=120); plt.show()


## 4. Препроцессинг: очистка, признаки, сплиты

In [ ]:
DROP_COLS = [
    "id", "project_id", "project_name",
    "price_per_sqft_usd", "price_per_m2_usd",
    "date_listed",
]

NUM_COLS = [
    "bedrooms", "area_sqft", "area_m2",
    "lat", "lon", "metro_distance_min", "to_burj_khalifa_km",
    "launch_year", "handover_year", "total_project_units",
    "mortgage_rate_at_listing",
    "project_age_at_listing", "years_to_handover",
    "year", "month", "months_since_start",
    "cbuae_base_rate_pct",
]

CAT_COLS = [
    "developer", "developer_tier",
    "community", "zone",
    "property_category", "property_type",
    "view", "payment_plan",
    "metro_station", "metro_line",
]


def _clean_offplan(off):
    df = off.copy()
    df["date_listed"] = pd.to_datetime(df["date_listed"], errors="coerce")
    df = df.dropna(subset=["date_listed", TARGET_RAW])
    df = df[df[TARGET_RAW] > 0].copy()

    df = df[(df["area_sqft"] > 0) & (df["area_m2"] > 0)]
    df = df[df["bedrooms"].between(0, 10)]
    df = df[df["launch_year"].between(1990, 2030)]
    df = df[df["handover_year"].between(1990, 2040)]
    df = df[(df["metro_distance_min"] >= 0) & (df["metro_distance_min"] < 1000)]
    df = df[(df["to_burj_khalifa_km"] >= 0) & (df["to_burj_khalifa_km"] < 100)]

    lo, hi = df[TARGET_RAW].quantile([0.001, 0.999])
    df = df[df[TARGET_RAW].between(lo, hi)]
    return df.reset_index(drop=True)


def _enrich_with_market(off, area):
    a = area[["year_month", "cbuae_base_rate_pct"]].drop_duplicates("year_month")
    off = off.copy()
    off["year_month"] = off["date_listed"].dt.strftime("%Y-%m")
    off = off.merge(a, on="year_month", how="left")
    off["cbuae_base_rate_pct"] = off["cbuae_base_rate_pct"].fillna(
        off["cbuae_base_rate_pct"].median())
    return off


def engineer_features(off, area):
    df = _clean_offplan(off)
    df = _enrich_with_market(df, area)

    df["year"]  = df["date_listed"].dt.year
    df["month"] = df["date_listed"].dt.month
    start = df["date_listed"].min()
    df["months_since_start"] = (
        (df["date_listed"].dt.year - start.year) * 12
        + df["date_listed"].dt.month - start.month
    )
    df["project_age_at_listing"] = df["date_listed"].dt.year - df["launch_year"]
    df["years_to_handover"]      = df["handover_year"] - df["date_listed"].dt.year

    df[TARGET] = np.log1p(df[TARGET_RAW])
    return df


def time_split(df):
    tr = df[df["date_listed"] <= TRAIN_END]
    va = df[(df["date_listed"] > TRAIN_END) & (df["date_listed"] <= VAL_END)]
    te = df[(df["date_listed"] > VAL_END)   & (df["date_listed"] <= TEST_END)]
    return tr, va, te


def random_split(df, seed=RANDOM_STATE):
    tr, tmp = train_test_split(df, test_size=0.30, random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.50, random_state=seed)
    return tr, va, te


def build_preprocessor(kind):
    if kind in ("linear", "mlp"):
        num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                             ("sc",  StandardScaler())])
        cat_pipe = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                             ("ohe", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))])
    else:  # tree
        num_pipe = Pipeline([("imp", SimpleImputer(strategy="median"))])
        cat_pipe = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                             ("ord", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                    unknown_value=-1))])
    return ColumnTransformer([("num", num_pipe, NUM_COLS),
                              ("cat", cat_pipe, CAT_COLS)], remainder="drop")


def get_Xy(df):
    X = df[NUM_COLS + CAT_COLS]
    return X, df[TARGET].values, df[TARGET_RAW].values


df = engineer_features(off, area)
print("после очистки:", df.shape)

train, val, test = time_split(df)
print("time split: train =", len(train), "val =", len(val), "test =", len(test))

X_tr, y_tr_log, y_tr_raw = get_Xy(train)
X_va, y_va_log, y_va_raw = get_Xy(val)
X_te, y_te_log, y_te_raw = get_Xy(test)


## 5. Определение моделей

In [ ]:
def mape(y_true, y_pred):
    mask = y_true > 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0


def evaluate(y_true_raw, y_pred_raw):
    return {
        "R2":   r2_score(y_true_raw, y_pred_raw),
        "MAE":  mean_absolute_error(y_true_raw, y_pred_raw),
        "RMSE": np.sqrt(mean_squared_error(y_true_raw, y_pred_raw)),
        "MAPE": mape(y_true_raw, y_pred_raw),
    }


def make_linear(kind="ols"):
    est = LinearRegression() if kind == "ols" else Ridge(alpha=1.0, random_state=RANDOM_STATE)
    return Pipeline([("prep", build_preprocessor("linear")), ("model", est)])


def make_rf():
    return Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                                        n_jobs=-1, random_state=RANDOM_STATE)),
    ])


class LightGBMWrapper:
    """LightGBM с нативными категориальными признаками pandas category."""

    def __init__(self, **params):
        self.model = LGBMRegressor(
            objective="regression",
            n_estimators=3000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )
        self.cat_categories = {}
        self.num_medians = {}

    def _prepare_features(self, X, fit=False):
        X = X[NUM_COLS + CAT_COLS].copy()

        # Числовые пропуски заполняем медианами, рассчитанными только на train.
        for col in NUM_COLS:
            X[col] = pd.to_numeric(X[col], errors="coerce")
            if fit:
                median = X[col].median()
                self.num_medians[col] = float(median) if pd.notna(median) else 0.0
            X[col] = X[col].fillna(self.num_medians[col])

        # Категории кодируются LightGBM нативно; неизвестные категории на predict
        # становятся NaN и обрабатываются моделью как пропущенные значения.
        for col in CAT_COLS:
            values = X[col].astype("string").fillna("__MISSING__")
            if fit:
                self.cat_categories[col] = pd.Index(values.unique())
            X[col] = pd.Categorical(values, categories=self.cat_categories[col])

        return X

    def fit(self, X, y, X_val=None, y_val=None):
        X_train = self._prepare_features(X, fit=True)
        fit_params = {"categorical_feature": "auto"}

        if X_val is not None and y_val is not None:
            X_valid = self._prepare_features(X_val)
            fit_params["eval_set"] = [(X_valid, y_val)]
            fit_params["eval_metric"] = "l1"
            fit_params["callbacks"] = [early_stopping(stopping_rounds=100, verbose=False)]

        self.model.fit(X_train, y, **fit_params)
        return self

    def predict(self, X):
        X_prepared = self._prepare_features(X)
        return self.model.predict(X_prepared)


def build_mlp(input_dim, arch=(64, 32)):
    from tensorflow.keras import layers, models, optimizers
    m = models.Sequential()
    m.add(layers.Input(shape=(input_dim,)))
    for h in arch:
        m.add(layers.Dense(h, activation="relu"))
        m.add(layers.Dropout(0.15))
    m.add(layers.Dense(1, activation="linear"))
    m.compile(optimizer=optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return m


class MLPWrapper:
    def __init__(self, arch=(64, 32)):
        self.arch = arch
        self.prep = build_preprocessor("mlp")
        self.model = None

    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch_size=128):
        Xt = self.prep.fit_transform(X)
        self.model = build_mlp(Xt.shape[1], self.arch)
        from tensorflow.keras import callbacks
        cb = [callbacks.EarlyStopping(patience=15, restore_best_weights=True,
                                      monitor="val_loss")]
        val = (self.prep.transform(X_val), y_val) if X_val is not None else None
        self.model.fit(Xt, y, validation_data=val,
                       epochs=epochs, batch_size=batch_size, verbose=0, callbacks=cb)
        return self

    def predict(self, X):
        return self.model.predict(self.prep.transform(X), verbose=0).ravel()


def train_all(X_tr, y_tr, X_va, y_va, mlp_arch=(128, 64, 32)):
    res = {}
    print("[Linear]");       res["Linear"] = make_linear("ols").fit(X_tr, y_tr)
    print("[Ridge]");        res["Ridge"] = make_linear("ridge").fit(X_tr, y_tr)
    print("[RandomForest]"); res["RandomForest"] = make_rf().fit(X_tr, y_tr)
    print("[LightGBM]")
    res["LightGBM"] = LightGBMWrapper().fit(X_tr, y_tr, X_va, y_va)
    print("[MLP]", mlp_arch)
    res["MLP"] = MLPWrapper(arch=mlp_arch).fit(X_tr, y_tr, X_va, y_va)
    return res


def validate_and_test(models, X_va, y_va_raw, X_te, y_te_raw):
    rows = []
    for name, m in models.items():
        pv = np.expm1(np.clip(m.predict(X_va), 0, None))
        pt = np.expm1(np.clip(m.predict(X_te), 0, None))
        rows.append({"model": name, "split": "val",  **evaluate(y_va_raw, pv)})
        rows.append({"model": name, "split": "test", **evaluate(y_te_raw, pt)})
    return pd.DataFrame(rows)

## 6. Обучение и метрики

In [ ]:
models = train_all(X_tr, y_tr_log, X_va, y_va_log, mlp_arch=(128, 64, 32))

metrics = validate_and_test(models, X_va, y_va_raw, X_te, y_te_raw)
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
metrics


## 7. Устойчивость к шумам
- (а) шум в **признаках** теста (feature + category + missing) → рост MAE;
- (б) шум в **цене train** → качество на чистом тесте.

In [ ]:
def _rng(seed):
    return np.random.default_rng(seed)


def inject_feature_noise(X, share, rng):
    X = X.copy(); n = len(X); k = int(round(n * share))
    if k == 0: return X
    idx = rng.choice(n, size=k, replace=False)
    X.iloc[idx, X.columns.get_loc("area_sqft")] *= rng.uniform(0.7, 1.3, size=k)
    X.iloc[idx, X.columns.get_loc("area_m2")]   *= rng.uniform(0.7, 1.3, size=k)
    X.iloc[idx, X.columns.get_loc("metro_distance_min")] += rng.normal(0, 10, size=k)
    X.iloc[idx, X.columns.get_loc("to_burj_khalifa_km")] += rng.normal(0, 2.0, size=k)
    X.iloc[idx, X.columns.get_loc("lat")] += rng.normal(0, 0.005, size=k)
    X.iloc[idx, X.columns.get_loc("lon")] += rng.normal(0, 0.005, size=k)
    X["area_sqft"] = X["area_sqft"].clip(lower=1)
    X["area_m2"]   = X["area_m2"].clip(lower=1)
    X["metro_distance_min"] = X["metro_distance_min"].clip(lower=0)
    X["to_burj_khalifa_km"] = X["to_burj_khalifa_km"].clip(lower=0)
    return X


def inject_category_noise(X, share, rng):
    X = X.copy(); n = len(X); k = int(round(n * share))
    for col in CAT_COLS:
        pool = X[col].dropna().unique()
        if k == 0 or len(pool) < 2: continue
        idx = rng.choice(n, size=k, replace=False)
        X.loc[X.index[idx], col] = rng.choice(pool, size=k)
    return X


def inject_missing(X, share, rng):
    X = X.copy(); n, m = X.shape
    total = int(round(n * m * share))
    if total == 0: return X
    rows = rng.integers(0, n, size=total); cols = rng.integers(0, m, size=total)
    for r, c in zip(rows, cols):
        X.iat[r, c] = np.nan
    return X


def corrupt_train_prices(y_log, share, rng):
    y = y_log.copy(); k = int(round(len(y) * share))
    if k == 0: return y
    idx = rng.choice(len(y), size=k, replace=False)
    y[idx] = np.log1p(np.expm1(y[idx]) * rng.uniform(0.5, 1.5, size=k))
    return y


def robustness_features(models, X_te, y_te_raw,
                        levels=NOISE_LEVELS, repeats=NOISE_REPEATS):
    rows = []
    for name, m in models.items():
        base = evaluate(y_te_raw, np.expm1(np.clip(m.predict(X_te), 0, None)))
        for lvl in levels:
            maes = []
            for r in range(repeats):
                rng = _rng(RANDOM_STATE + r + int(lvl * 1000))
                Xn = inject_feature_noise(X_te, lvl, rng)
                Xn = inject_category_noise(Xn, lvl, rng)
                Xn = inject_missing(Xn, lvl, rng)
                pred = np.expm1(np.clip(m.predict(Xn), 0, None))
                maes.append(evaluate(y_te_raw, pred)["MAE"])
            mae_n = float(np.mean(maes))
            rows.append({"model": name, "noise": lvl,
                         "MAE_clean": base["MAE"], "MAE_noisy": mae_n,
                         "rel_growth_MAE": (mae_n - base["MAE"]) / base["MAE"]})
    return pd.DataFrame(rows)


def price_noise_in_train(models, X_tr, y_tr_log, X_te, y_te_raw, levels=NOISE_LEVELS):
    rows = []
    for lvl in levels:
        rng = _rng(RANDOM_STATE + int(lvl * 1000))
        y_c = corrupt_train_prices(y_tr_log, lvl, rng)
        for name, m in models.items():
            m2 = copy.deepcopy(m)
            try:
                m2.fit(X_tr, y_c)
            except TypeError:
                m2.fit(X_tr, y_c, X_tr, y_c)
            pred = np.expm1(np.clip(m2.predict(X_te), 0, None))
            rows.append({"model": name, "price_noise": lvl, **evaluate(y_te_raw, pred)})
    return pd.DataFrame(rows)


rob = robustness_features(models, X_te, y_te_raw)
rob.to_csv(OUT_DIR / "robustness_features.csv", index=False)
print("=== ROBUSTNESS (features) ==="); print(rob.head(15))

pn = price_noise_in_train(models, X_tr, y_tr_log, X_te, y_te_raw)
pn.to_csv(OUT_DIR / "robustness_prices.csv", index=False)
print("\n=== PRICE NOISE (train) ==="); print(pn.head(15))


## 8. Сценарии (по Ясницкому): ипотечная ставка, площадь, метро

In [ ]:
def _predict_raw(model, X):
    return np.expm1(np.clip(model.predict(X), 0, None))


Xs = X_te.sample(n=min(500, len(X_te)), random_state=RANDOM_STATE)

def scenario_rate(models, X_base, rates=(1.5, 3.0, 4.65, 5.5, 6.9)):
    rows = []
    for name, m in models.items():
        for r in rates:
            X = X_base.copy(); X["mortgage_rate_at_listing"] = r
            rows.append({"model": name, "rate": r,
                         "median_pred": float(np.median(_predict_raw(m, X)))})
    return pd.DataFrame(rows)

def scenario_area(models, X_base, factor=(0.5, 0.75, 1.0, 1.25, 1.5)):
    rows = []
    for name, m in models.items():
        for f in factor:
            X = X_base.copy()
            X["area_sqft"] = X["area_sqft"] * f
            X["area_m2"]   = X["area_m2"]   * f
            rows.append({"model": name, "area_factor": f,
                         "median_pred": float(np.median(_predict_raw(m, X)))})
    return pd.DataFrame(rows)

def scenario_metro(models, X_base, distances=(5, 10, 20, 30, 45)):
    rows = []
    for name, m in models.items():
        for d in distances:
            X = X_base.copy(); X["metro_distance_min"] = d
            rows.append({"model": name, "metro_min": d,
                         "median_pred": float(np.median(_predict_raw(m, X)))})
    return pd.DataFrame(rows)

scenario_rate(models, Xs).to_csv(OUT_DIR / "scen_rate.csv", index=False)
scenario_area(models, Xs).to_csv(OUT_DIR / "scen_area.csv", index=False)
scenario_metro(models, Xs).to_csv(OUT_DIR / "scen_metro.csv", index=False)
print("Сценарии сохранены.")


## 9. Интерпретация
- коэффициенты линейной модели;
- permutation importance для RF;
- worst errors для LightGBM.

In [ ]:
# (1) Коэффициенты линейной модели
prep = models["Linear"].named_steps["prep"]
feats = prep.get_feature_names_out()
coefs = pd.Series(models["Linear"].named_steps["model"].coef_,
                  index=feats).sort_values(key=np.abs, ascending=False)
coefs.to_csv(OUT_DIR / "coefs_linear.csv")
print("=== Top-15 |coef| (Linear) ==="); print(coefs.head(15))

# (2) Permutation importance (RF)
pi = permutation_importance(models["RandomForest"], X_te, y_te_log,
                            n_repeats=10, random_state=RANDOM_STATE,
                            scoring="neg_mean_absolute_error")
imp = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
imp.to_csv(OUT_DIR / "perm_importance_rf.csv")
print("\n=== Perm importance (RF) top-15 ==="); print(imp.head(15))

# (3) Worst errors (LightGBM)
pred_lgbm = np.expm1(np.clip(models["LightGBM"].predict(X_te), 0, None))
err = np.abs(test[TARGET_RAW].values - pred_lgbm)
worst = (test.reset_index(drop=True).assign(abs_error=err)
             .nlargest(20, "abs_error")
             [["community", "zone", "property_category", "property_type",
               "price_usd", "abs_error", "date_listed"]])
worst.to_csv(OUT_DIR / "worst_errors_lightgbm.csv", index=False)
print("\n=== Worst errors (LightGBM) ==="); worst.head(10)

## 10. Готово

Все артефакты — в `outputs/`:
- `metrics.csv`, `robustness_features.csv`, `robustness_prices.csv`;
- `scen_rate.csv`, `scen_area.csv`, `scen_metro.csv`;
- `coefs_linear.csv`, `perm_importance_rf.csv`, `worst_errors_lightgbm.csv`;
- PNG-графики из EDA.